## RAG Day 4

### Evaluation!

This is such an important part of building an accurate and reliable RAG pipeline — and it applies to many ways of solving business problems with LLMs.

People often obsess over RAG architecture and frameworks. But even more important: **evaluations**.

We evaluate our pipeline on two fronts:
1. **Retrieval** — did we pull the *right context* from the vector store? (MRR, nDCG, keyword coverage)
2. **Answer quality** — is the final generated answer accurate, complete, and relevant? (LLM-as-a-judge)

We reuse the VectorStore we built earlier (with OpenAI `text-embedding-3-large`). The query embedding model **must** match the one used to build the store.

In [1]:
import json
import math
from pathlib import Path
from collections import Counter

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.messages import SystemMessage, HumanMessage

/Users/marcolerma/Personal/GitHub-marclerm/applied-llm-engineering/.venv/lib/python3.14/site-packages/langchain_core/utils/pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1
/Users/marcolerma/Personal/GitHub-marclerm/applied-llm-engineering/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL = "gpt-4.1-nano"
RETRIEVAL_K = 10  # how many chunks to retrieve per query

for candidate in [Path.cwd(), *Path.cwd().parents]:
    possible_path = candidate / "lectures" / "week-five" / "vector_db"
    if possible_path.exists():
        DB_NAME = str(possible_path)
        TESTS_FILE = str(possible_path.parent / "tests.jsonl")
        break
else:
    DB_NAME = "vector_db"
    TESTS_FILE = "tests.jsonl"
    print("No lectures/week-five/vector_db found yet. Run rag-vector-store-visualization.ipynb first to create it.")

load_dotenv(override=True)

True

### Connect to Chroma and set up the RAG pipeline (same as Day 3)

Embeddings must match how the store was built: OpenAI `text-embedding-3-large`.

In [4]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": RETRIEVAL_K})
llm = ChatOpenAI(temperature=0, model_name=MODEL)

In [5]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""


def fetch_context(question: str) -> list[Document]:
    """Retrieve the relevant context documents for a question."""
    return retriever.invoke(question)


def answer_question(question: str) -> tuple[str, list[Document]]:
    """Answer the question with RAG; return the answer AND the context documents."""
    docs = fetch_context(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content, docs

### The test set

We've prepared a set of 150 questions in `tests.jsonl`. Each test has:
- **question** — what we ask the RAG system
- **keywords** — terms that *must* appear in the retrieved context (used to score retrieval)
- **reference_answer** — the gold answer (used to score the generated answer)
- **category** — the kind of question (`direct_fact`, `temporal`, `spanning`, ...)

In [6]:
class TestQuestion(BaseModel):
    """A test question with expected keywords and a reference answer."""

    question: str = Field(description="The question to ask the RAG system")
    keywords: list[str] = Field(description="Keywords that must appear in retrieved context")
    reference_answer: str = Field(description="The reference answer for this question")
    category: str = Field(description="Question category (e.g., direct_fact, spanning, temporal)")


def load_tests() -> list[TestQuestion]:
    """Load test questions from the JSONL file."""
    tests = []
    with open(TESTS_FILE, "r", encoding="utf-8") as f:
        for line in f:
            tests.append(TestQuestion(**json.loads(line.strip())))
    return tests

In [7]:
tests = load_tests()
len(tests)

150

In [8]:
example = tests[0]
print(example.question)
print(example.category)
print(example.reference_answer)
print(example.keywords)

Who won the prestigious IIOTY award in 2023?
direct_fact
Maxine Thompson won the prestigious Insurellm Innovator of the Year (IIOTY) award in 2023.
['Maxine', 'Thompson', 'IIOTY']


In [9]:
# How many of each category of question do we have?
Counter(t.category for t in tests)

Counter({'direct_fact': 70,
         'temporal': 20,
         'spanning': 20,
         'comparative': 10,
         'numerical': 10,
         'relationship': 10,
         'holistic': 10})

## Part 1: Evaluating Retrieval

Before we judge the answer, we should ask: did the retriever even surface the right context?

We use the test's **keywords** as a proxy for "relevant". Three classic information-retrieval metrics:

- **Keyword coverage** — what fraction of the keywords appeared *anywhere* in the top-k chunks.
- **MRR (Mean Reciprocal Rank)** — rewards finding a keyword *early*. If a keyword first appears in chunk #1 you score 1.0, in chunk #2 you score 1/2, chunk #3 → 1/3, etc. We average across keywords.
- **nDCG (Normalized Discounted Cumulative Gain)** — also rewards relevant chunks ranking high, with a logarithmic discount, normalized so the perfect ranking scores 1.0.

All comparisons are case-insensitive.

In [10]:
def calculate_mrr(keyword: str, retrieved_docs: list[Document]) -> float:
    """Reciprocal rank for a single keyword: 1/rank of the first chunk that contains it."""
    keyword_lower = keyword.lower()
    for rank, doc in enumerate(retrieved_docs, start=1):
        if keyword_lower in doc.page_content.lower():
            return 1.0 / rank
    return 0.0


def calculate_dcg(relevances: list[int], k: int) -> float:
    """Discounted Cumulative Gain."""
    dcg = 0.0
    for i in range(min(k, len(relevances))):
        dcg += relevances[i] / math.log2(i + 2)  # i+2 because rank starts at 1
    return dcg


def calculate_ndcg(keyword: str, retrieved_docs: list[Document], k: int = RETRIEVAL_K) -> float:
    """nDCG for a single keyword using binary relevance (keyword present / absent)."""
    keyword_lower = keyword.lower()
    relevances = [1 if keyword_lower in doc.page_content.lower() else 0 for doc in retrieved_docs[:k]]
    dcg = calculate_dcg(relevances, k)
    # Ideal DCG: the best case is the keyword appearing in the first position
    idcg = calculate_dcg(sorted(relevances, reverse=True), k)
    return dcg / idcg if idcg > 0 else 0.0

In [12]:
class RetrievalEval(BaseModel):
    """Evaluation metrics for retrieval performance."""

    mrr: float = Field(description="Mean Reciprocal Rank - averaged across all keywords")
    ndcg: float = Field(description="Normalized Discounted Cumulative Gain (binary relevance)")
    keywords_found: int = Field(description="Number of keywords found in the top-k results")
    total_keywords: int = Field(description="Total number of keywords to find")
    keyword_coverage: float = Field(description="Percentage of keywords found")


def evaluate_retrieval(test: TestQuestion, k: int = RETRIEVAL_K) -> RetrievalEval:
    """Evaluate retrieval performance for a single test question."""
    retrieved_docs = fetch_context(test.question)

    mrr_scores = [calculate_mrr(kw, retrieved_docs) for kw in test.keywords]
    avg_mrr = sum(mrr_scores) / len(mrr_scores) if mrr_scores else 0.0

    ndcg_scores = [calculate_ndcg(kw, retrieved_docs, k) for kw in test.keywords]
    avg_ndcg = sum(ndcg_scores) / len(ndcg_scores) if ndcg_scores else 0.0

    keywords_found = sum(1 for score in mrr_scores if score > 0)
    total_keywords = len(test.keywords)
    keyword_coverage = (keywords_found / total_keywords * 100) if total_keywords else 0.0

    return RetrievalEval(
        mrr=avg_mrr,
        ndcg=avg_ndcg,
        keywords_found=keywords_found,
        total_keywords=total_keywords,
        keyword_coverage=keyword_coverage,
    )

In [13]:
evaluate_retrieval(example)

RetrievalEval(mrr=0.6666666666666666, ndcg=0.5561086263536886, keywords_found=2, total_keywords=3, keyword_coverage=66.66666666666666)

## Part 2: Evaluating the Answer — LLM as a Judge

Retrieval might be perfect but the final answer could still be wrong. To score the generated answer we use a powerful technique: **LLM-as-a-judge**.

We give an LLM the question, our generated answer, and the reference answer, and ask it to score on three dimensions (1–5):
- **Accuracy** — factually correct vs. the reference? (any wrong answer scores 1)
- **Completeness** — does it cover everything in the reference answer?
- **Relevance** — does it answer the question directly, with no fluff?

We use LangChain's structured output (`with_structured_output`) so the judge returns a typed `AnswerEval` object.

In [14]:
class AnswerEval(BaseModel):
    """LLM-as-a-judge evaluation of answer quality."""

    feedback: str = Field(description="Concise feedback comparing the answer to the reference answer")
    accuracy: float = Field(description="Factual correctness vs reference. 1 (wrong) to 5 (perfect). 3 is acceptable.")
    completeness: float = Field(description="Covers all aspects of the question. 1 (missing key info) to 5 (all reference info included).")
    relevance: float = Field(description="Directly addresses the question with no extra info. 1 (off-topic) to 5 (perfectly relevant).")


judge = ChatOpenAI(temperature=0, model_name=MODEL).with_structured_output(AnswerEval)

JUDGE_SYSTEM_PROMPT = (
    "You are an expert evaluator assessing the quality of answers. "
    "Evaluate the generated answer by comparing it to the reference answer. "
    "Only give 5/5 scores for perfect answers."
)

In [15]:
def evaluate_answer(test: TestQuestion) -> tuple[AnswerEval, str, list[Document]]:
    """Evaluate answer quality using LLM-as-a-judge."""
    generated_answer, retrieved_docs = answer_question(test.question)

    judge_prompt = f"""Question:
{test.question}

Generated Answer:
{generated_answer}

Reference Answer:
{test.reference_answer}

Please evaluate the generated answer on three dimensions:
1. Accuracy: How factually correct is it compared to the reference answer? Only give 5/5 for perfect answers. If the answer is wrong, accuracy must be 1.
2. Completeness: How thoroughly does it cover all the information from the reference answer?
3. Relevance: How directly does it answer the question, giving no additional information?

Provide detailed feedback and scores from 1 (very poor) to 5 (ideal) for each dimension."""

    answer_eval = judge.invoke([
        SystemMessage(content=JUDGE_SYSTEM_PROMPT),
        HumanMessage(content=judge_prompt),
    ])
    return answer_eval, generated_answer, retrieved_docs

In [16]:
eval, answer, chunks = evaluate_answer(example)
answer

'Maxine was recognized as Insurellm Innovator of the Year (IIOTY) in 2023.'

In [17]:
eval

AnswerEval(feedback="The answer correctly identifies the recipient as Maxine and the award as Insurellm Innovator of the Year (IIOTY) in 2023, matching the reference. However, it omits the full name 'Maxine Thompson,' which is present in the reference. The answer directly addresses the question and is relevant, but slightly less complete due to the missing full name.", accuracy=4.0, completeness=3.0, relevance=5.0)

In [18]:
print(eval.feedback)
print("Accuracy:", eval.accuracy)
print("Completeness:", eval.completeness)
print("Relevance:", eval.relevance)

The answer correctly identifies the recipient as Maxine and the award as Insurellm Innovator of the Year (IIOTY) in 2023, matching the reference. However, it omits the full name 'Maxine Thompson,' which is present in the reference. The answer directly addresses the question and is relevant, but slightly less complete due to the missing full name.
Accuracy: 4.0
Completeness: 3.0
Relevance: 5.0


## Putting it together: evaluate across the whole test set

Now we can run our metrics over many questions and look at averages — overall and broken down by category. This is how you'd track whether a change (different chunk size, embedding model, value of `k`, prompt tweaks...) actually *improves* your pipeline.

_Tip: running all 150 tests calls the LLM judge 150 times. Start with a small slice while iterating._

In [19]:
sample = tests[:10]  # bump this up (e.g. `tests`) for a full run

results = []
for i, test in enumerate(sample, start=1):
    retrieval = evaluate_retrieval(test)
    answer_eval, _, _ = evaluate_answer(test)
    results.append((test, retrieval, answer_eval))
    print(f"[{i}/{len(sample)}] {test.category:12s} | coverage {retrieval.keyword_coverage:5.1f}% | "
          f"acc {answer_eval.accuracy:.0f} comp {answer_eval.completeness:.0f} rel {answer_eval.relevance:.0f}")

[1/10] direct_fact  | coverage  66.7% | acc 5 comp 4 rel 5
[2/10] direct_fact  | coverage 100.0% | acc 4 comp 4 rel 5
[3/10] direct_fact  | coverage 100.0% | acc 5 comp 5 rel 5
[4/10] direct_fact  | coverage 100.0% | acc 4 comp 3 rel 5
[5/10] direct_fact  | coverage 100.0% | acc 5 comp 5 rel 5
[6/10] direct_fact  | coverage 100.0% | acc 5 comp 3 rel 5
[7/10] direct_fact  | coverage 100.0% | acc 5 comp 5 rel 5
[8/10] direct_fact  | coverage 100.0% | acc 5 comp 4 rel 5
[9/10] direct_fact  | coverage 100.0% | acc 5 comp 5 rel 5
[10/10] direct_fact  | coverage 100.0% | acc 4 comp 3 rel 5


In [20]:
n = len(results)
print("=== Averages across", n, "tests ===")
print(f"MRR:              {sum(r.mrr for _, r, _ in results) / n:.3f}")
print(f"nDCG:             {sum(r.ndcg for _, r, _ in results) / n:.3f}")
print(f"Keyword coverage: {sum(r.keyword_coverage for _, r, _ in results) / n:.1f}%")
print(f"Accuracy:         {sum(a.accuracy for _, _, a in results) / n:.2f}/5")
print(f"Completeness:     {sum(a.completeness for _, _, a in results) / n:.2f}/5")
print(f"Relevance:        {sum(a.relevance for _, _, a in results) / n:.2f}/5")

=== Averages across 10 tests ===
MRR:              0.871
nDCG:             0.832
Keyword coverage: 96.7%
Accuracy:         4.70/5
Completeness:     4.10/5
Relevance:        5.00/5


## The takeaway

Evaluation turns RAG from guesswork into engineering. With retrieval metrics and an LLM judge in place, you can confidently answer: *"did that change make things better or worse?"* — the most important question in any business RAG project. 🚀